# Allen Visual Behavior Cohort and Session Selection

Explore Allen Visual Behavior Ophys metadata, define a cohort, summarize behavioral performance, and rank candidate sessions for downstream analysis. This is a standalone Colab notebook: open or upload this `.ipynb` in Google Colab and run it from top to bottom. It installs its own dependencies and downloads metadata through AllenSDK, so cloning this repository, mounting Google Drive, and supplying local project files are not required.

For comparison only, `01_colab_drive_parquet_demo.ipynb` demonstrates shared-parquet loading, while `02_colab_drive_hit_miss_pca.ipynb` loads that parquet for hit-versus-miss state-space analysis. Those notebooks are not dependencies of this notebook. This notebook loads AllenSDK project metadata directly because its experiment, cell, behavior-session, and ophys-session tables are not all available from their analysis parquet.


In [ ]:
# PyPI release 2.16.2 predates Python 3.12/3.13 support.
# Install the upstream compatibility fix merged by the Allen Institute.
%pip install -q --upgrade pip setuptools wheel
%pip install -q "allensdk @ git+https://github.com/AllenInstitute/AllenSDK.git@05328c2f4f4079af4df2325bea9ebf143922b875"


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from IPython.display import display

In [ ]:
def count_table(dataframe, column):
    """
    Count the values and percentages of a categorical column.
    """
    if column not in dataframe.columns:
        return pd.DataFrame()

    counts = dataframe[column].value_counts(
        dropna=False
    )

    output = pd.DataFrame({
        "n": counts,
        "percent": counts / counts.sum() * 100
    })

    output.index.name = column

    return output


def plot_count_table(
    table,
    title,
    xlabel=None
):
    """
    Plot category counts as a bar chart.
    """
    if table.empty:
        print(f"No data available for: {title}")
        return

    table_to_plot = table.sort_values(
        "n",
        ascending=False
    )

    plt.figure(figsize=(8, 5))

    plt.bar(
        table_to_plot.index.astype(str),
        table_to_plot["n"]
    )

    plt.title(title)
    plt.xlabel(xlabel or table.index.name)
    plt.ylabel("Number of experiments")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_distribution(
    values,
    title,
    xlabel,
    bins=20
):
    """
    Plot the distribution of continuous values as a histogram.
    """
    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).dropna()

    if len(values) == 0:
        print(f"No valid data for: {title}")
        return

    plt.figure(figsize=(7, 5))
    plt.hist(values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Number of sessions")
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import sys
from pathlib import Path

from allensdk.brain_observatory.behavior.behavior_project_cache import (
    VisualBehaviorOphysProjectCache,
)

# Use Colab's temporary storage by default; no repository checkout is needed.
IN_COLAB = "google.colab" in sys.modules
DEFAULT_CACHE_DIR = Path("/content/allen_visual_behavior") if IN_COLAB else Path("data/cache/allensdk")
# Optionally override this path, for example after mounting persistent Drive storage.
CACHE_DIR = Path(os.environ.get("ALLENSDK_CACHE_DIR", DEFAULT_CACHE_DIR))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

cache = VisualBehaviorOphysProjectCache.from_s3_cache(cache_dir=str(CACHE_DIR))

print("Cache initialized successfully.")
print("Cache directory:", CACHE_DIR.resolve())


In [ ]:
experiment_table = (
    cache.get_ophys_experiment_table()
    .copy()
)

cells_table = (
    cache.get_ophys_cells_table()
    .copy()
)

behavior_session_table = (
    cache.get_behavior_session_table()
    .copy()
)

ophys_session_table = (
    cache.get_ophys_session_table()
    .copy()
)

print("Experiment table:", experiment_table.shape)
print("Cells table:", cells_table.shape)
print("Behavior session table:", behavior_session_table.shape)
print("Ophys session table:", ophys_session_table.shape)

In [ ]:
overall_summary = pd.Series({
    "n_ophys_experiments":
        len(experiment_table),

    "n_ophys_sessions":
        experiment_table[
            "ophys_session_id"
        ].nunique(),

    "n_behavior_sessions":
        experiment_table[
            "behavior_session_id"
        ].nunique(),

    "n_mice":
        experiment_table[
            "mouse_id"
        ].nunique(),

    "n_cells":
        len(cells_table),

    "n_cre_lines":
        experiment_table[
            "cre_line"
        ].nunique(),

    "n_targeted_structures":
        experiment_table[
            "targeted_structure"
        ].nunique(),

    "n_project_codes":
        experiment_table[
            "project_code"
        ].nunique(),
}, name="value")

display(
    overall_summary.to_frame()
)

In [ ]:
cre_line_counts = count_table(
    experiment_table,
    "cre_line"
)

display(cre_line_counts)

In [ ]:
plot_count_table(
    cre_line_counts,
    title="Experiments by cell type",
    xlabel="Cre line"
)

In [ ]:
structure_counts = count_table(
    experiment_table,
    "targeted_structure"
)

display(structure_counts)

In [ ]:
plot_count_table(
    structure_counts,
    title="Experiments by targeted structure",
    xlabel="Brain area"
)

In [ ]:
experience_counts = count_table(
    experiment_table,
    "experience_level"
)

display(experience_counts)

In [ ]:
plot_count_table(
    experience_counts,
    title="Experiments by experience level",
    xlabel="Experience level"
)

In [ ]:
passive_counts = count_table(
    experiment_table,
    "passive"
)

display(passive_counts)

In [ ]:
plot_count_table(
    passive_counts,
    title="Active and passive experiments",
    xlabel="Passive"
)

In [ ]:
project_counts = count_table(
    experiment_table,
    "project_code"
)

display(project_counts)

In [ ]:
plot_count_table(
    project_counts,
    title="Experiments by project",
    xlabel="Project code"
)

In [ ]:
cell_counts = (
    cells_table
    .groupby("ophys_experiment_id")
    .size()
    .rename("n_cells")
)

experiment_enriched = (
    experiment_table
    .join(
        cell_counts,
        how="left"
    )
)

experiment_enriched["n_cells"] = (
    experiment_enriched["n_cells"]
    .fillna(0)
    .astype(int)
)

In [ ]:
plot_distribution(
    experiment_enriched["n_cells"],
    title="Number of cells per experiment",
    xlabel="Cells per experiment",
    bins=30
)

In [ ]:
cells_by_cre_line = (
    experiment_enriched
    .groupby("cre_line")[
        "n_cells"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
)

display(cells_by_cre_line)

In [ ]:
cre_lines = (
    experiment_enriched[
        "cre_line"
    ]
    .dropna()
    .unique()
)

boxplot_data = [
    experiment_enriched.loc[
        experiment_enriched[
            "cre_line"
        ] == cre_line,
        "n_cells"
    ]
    for cre_line in cre_lines
]

plt.figure(figsize=(7, 5))

plt.boxplot(
    boxplot_data,
    tick_labels=cre_lines,
    showfliers=False
)

plt.title(
    "Cells per experiment by cell type"
)

plt.xlabel("Cre line")
plt.ylabel("Number of cells")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
TARGET_CRE_LINE = "Vip-IRES-Cre"
TARGET_STRUCTURE = "VISp"
TARGET_EXPERIENCE = "Familiar"
TARGET_PASSIVE = False

In [ ]:
cohort_mask = (
    (
        experiment_enriched[
            "cre_line"
        ] == TARGET_CRE_LINE
    )
    & (
        experiment_enriched[
            "targeted_structure"
        ] == TARGET_STRUCTURE
    )
    & (
        experiment_enriched[
            "experience_level"
        ] == TARGET_EXPERIENCE
    )
    & (
        experiment_enriched[
            "passive"
        ] == TARGET_PASSIVE
    )
)

cohort_experiments = (
    experiment_enriched.loc[
        cohort_mask
    ]
    .copy()
)

print(
    "Experiments in target cohort:",
    len(cohort_experiments)
)

In [ ]:
cohort_summary = pd.Series({
    "n_experiments":
        len(cohort_experiments),

    "n_ophys_sessions":
        cohort_experiments[
            "ophys_session_id"
        ].nunique(),

    "n_behavior_sessions":
        cohort_experiments[
            "behavior_session_id"
        ].nunique(),

    "n_mice":
        cohort_experiments[
            "mouse_id"
        ].nunique(),

    "total_cells":
        cohort_experiments[
            "n_cells"
        ].sum(),

    "median_cells_per_experiment":
        cohort_experiments[
            "n_cells"
        ].median(),

    "mean_cells_per_experiment":
        cohort_experiments[
            "n_cells"
        ].mean(),
}, name="value")

display(
    cohort_summary.to_frame()
)

In [ ]:
cohort_project_counts = count_table(
    cohort_experiments,
    "project_code"
)

display(cohort_project_counts)

In [ ]:
cohort_session_type_counts = count_table(
    cohort_experiments,
    "session_type"
)

display(cohort_session_type_counts)

In [ ]:
plot_distribution(
    cohort_experiments["n_cells"],
    title=(
        "Cells per experiment: "
        "VIP-VISp-Familiar-active"
    ),
    xlabel="Cells per experiment",
    bins=20
)

In [ ]:
cohort_reset = (
    cohort_experiments
    .reset_index()
)

In [ ]:
session_imaging_summary = (
    cohort_reset
    .groupby(
        "behavior_session_id",
        as_index=False
    )
    .agg(
        mouse_id=(
            "mouse_id",
            "first"
        ),

        session_type=(
            "session_type",
            "first"
        ),

        project_code=(
            "project_code",
            "first"
        ),

        n_ophys_experiments=(
            "ophys_experiment_id",
            "nunique"
        ),

        total_cells_across_planes=(
            "n_cells",
            "sum"
        ),

        max_cells_in_one_plane=(
            "n_cells",
            "max"
        )
    )
)

In [ ]:
best_experiment_per_session = (
    cohort_reset
    .sort_values(
        "n_cells",
        ascending=False
    )
    .drop_duplicates(
        subset="behavior_session_id"
    )
    [
        [
            "behavior_session_id",
            "ophys_experiment_id",
            "imaging_depth",
            "n_cells"
        ]
    ]
    .rename(
        columns={
            "ophys_experiment_id":
                "representative_experiment_id",

            "n_cells":
                "representative_n_cells"
        }
    )
)

In [ ]:
session_imaging_summary = (
    session_imaging_summary
    .merge(
        best_experiment_per_session,
        on="behavior_session_id",
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
behavior_count_columns = [
    "trial_count",
    "go_trial_count",
    "catch_trial_count",
    "hit_trial_count",
    "miss_trial_count",
    "false_alarm_trial_count",
    "correct_reject_trial_count",
    "engaged_trial_count"
]

available_behavior_columns = [
    column
    for column in behavior_count_columns
    if column in behavior_session_table.columns
]

missing_behavior_columns = [
    column
    for column in behavior_count_columns
    if column not in behavior_session_table.columns
]

print(
    "Available behavior columns:",
    available_behavior_columns
)

print(
    "Missing behavior columns:",
    missing_behavior_columns
)

In [ ]:
behavior_metadata = (
    behavior_session_table[
        available_behavior_columns
    ]
    .reset_index()
)

In [ ]:
cohort_behavior = (
    session_imaging_summary
    .merge(
        behavior_metadata,
        on="behavior_session_id",
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
for column in available_behavior_columns:
    cohort_behavior[column] = pd.to_numeric(
        cohort_behavior[column],
        errors="coerce"
    )

In [ ]:
required_outcome_columns = [
    "hit_trial_count",
    "miss_trial_count",
    "false_alarm_trial_count",
    "correct_reject_trial_count"
]

missing_required = [
    column
    for column in required_outcome_columns
    if column not in cohort_behavior.columns
]

if missing_required:
    raise KeyError(
        "Behavior metadata is missing required columns: "
        f"{missing_required}"
    )

In [ ]:
cohort_behavior["n_change_trials"] = (
    cohort_behavior[
        "hit_trial_count"
    ]
    + cohort_behavior[
        "miss_trial_count"
    ]
)

cohort_behavior["n_catch_trials"] = (
    cohort_behavior[
        "false_alarm_trial_count"
    ]
    + cohort_behavior[
        "correct_reject_trial_count"
    ]
)

In [ ]:
cohort_behavior["hit_rate"] = (
    cohort_behavior[
        "hit_trial_count"
    ]
    / cohort_behavior[
        "n_change_trials"
    ]
)

cohort_behavior["false_alarm_rate"] = (
    cohort_behavior[
        "false_alarm_trial_count"
    ]
    / cohort_behavior[
        "n_catch_trials"
    ]
)

In [ ]:
cohort_behavior[
    "corrected_hit_rate"
] = (
    cohort_behavior[
        "hit_trial_count"
    ] + 0.5
) / (
    cohort_behavior[
        "n_change_trials"
    ] + 1
)

cohort_behavior[
    "corrected_false_alarm_rate"
] = (
    cohort_behavior[
        "false_alarm_trial_count"
    ] + 0.5
) / (
    cohort_behavior[
        "n_catch_trials"
    ] + 1
)

In [ ]:
cohort_behavior["d_prime"] = (
    norm.ppf(
        cohort_behavior[
            "corrected_hit_rate"
        ]
    )
    - norm.ppf(
        cohort_behavior[
            "corrected_false_alarm_rate"
        ]
    )
)

cohort_behavior["criterion"] = -0.5 * (
    norm.ppf(
        cohort_behavior[
            "corrected_hit_rate"
        ]
    )
    + norm.ppf(
        cohort_behavior[
            "corrected_false_alarm_rate"
        ]
    )
)

In [ ]:
cohort_behavior["minority_count"] = (
    cohort_behavior[
        [
            "hit_trial_count",
            "miss_trial_count"
        ]
    ]
    .min(axis=1)
)

In [ ]:
if (
    "engaged_trial_count"
    in cohort_behavior.columns
    and "trial_count"
    in cohort_behavior.columns
):
    cohort_behavior[
        "engaged_fraction"
    ] = (
        cohort_behavior[
            "engaged_trial_count"
        ]
        / cohort_behavior[
            "trial_count"
        ]
    )
else:
    cohort_behavior[
        "engaged_fraction"
    ] = np.nan

In [ ]:
behavior_metrics = [
    "trial_count",
    "hit_trial_count",
    "miss_trial_count",
    "false_alarm_trial_count",
    "correct_reject_trial_count",
    "hit_rate",
    "false_alarm_rate",
    "d_prime",
    "criterion",
    "engaged_fraction",
    "minority_count",
    "representative_n_cells"
]

behavior_metrics = [
    column
    for column in behavior_metrics
    if column in cohort_behavior.columns
]

In [ ]:
cohort_behavior_describe = (
    cohort_behavior[
        behavior_metrics
    ]
    .describe()
    .T
)

display(
    cohort_behavior_describe
)

In [ ]:
session_type_behavior_summary = (
    cohort_behavior
    .groupby("session_type")
    [
        [
            "hit_rate",
            "false_alarm_rate",
            "d_prime",
            "criterion",
            "engaged_fraction",
            "minority_count",
            "representative_n_cells"
        ]
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std"
    ])
)

display(
    session_type_behavior_summary
)

In [ ]:
plot_distribution(
    cohort_behavior[
        "false_alarm_rate"
    ],
    title="False-alarm-rate distribution",
    xlabel="False-alarm rate",
    bins=15
)

In [ ]:
plot_distribution(
    cohort_behavior["d_prime"],
    title="Behavioral sensitivity distribution",
    xlabel="d-prime",
    bins=15
)

In [ ]:
plot_distribution(
    cohort_behavior["criterion"],
    title="Response criterion distribution",
    xlabel="Criterion",
    bins=15
)

In [ ]:
plot_distribution(
    cohort_behavior[
        "engaged_fraction"
    ],
    title="Engaged-trial fraction",
    xlabel="Engaged fraction",
    bins=15
)

In [ ]:
valid_scatter = cohort_behavior[
    [
        "hit_trial_count",
        "miss_trial_count"
    ]
].dropna()

plt.figure(figsize=(6, 5))

plt.scatter(
    valid_scatter[
        "hit_trial_count"
    ],
    valid_scatter[
        "miss_trial_count"
    ]
)

plt.xlabel("Number of hit trials")
plt.ylabel("Number of miss trials")
plt.title("Hit and miss trial counts")

plt.tight_layout()
plt.show()

In [ ]:
valid_scatter = cohort_behavior[
    [
        "d_prime",
        "engaged_fraction"
    ]
].dropna()

plt.figure(figsize=(6, 5))

plt.scatter(
    valid_scatter[
        "engaged_fraction"
    ],
    valid_scatter[
        "d_prime"
    ]
)

plt.xlabel("Engaged fraction")
plt.ylabel("d-prime")
plt.title(
    "Behavioral sensitivity and engagement"
)

plt.tight_layout()
plt.show()

In [ ]:
cohort_behavior[
    "descriptive_candidate"
] = (
    (
        cohort_behavior[
            "hit_trial_count"
        ] >= 20
    )
    & (
        cohort_behavior[
            "miss_trial_count"
        ] >= 20
    )
    & (
        cohort_behavior[
            "representative_n_cells"
        ] >= 20
    )
    & (
        cohort_behavior[
            "d_prime"
        ] >= 0.5
    )
)

In [ ]:
candidate_sessions = (
    cohort_behavior
    .sort_values(
        [
            "descriptive_candidate",
            "d_prime",
            "minority_count",
            "representative_n_cells"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
)

In [ ]:
candidate_columns = [
    "behavior_session_id",
    "representative_experiment_id",
    "mouse_id",
    "session_type",
    "project_code",
    "representative_n_cells",
    "hit_trial_count",
    "miss_trial_count",
    "false_alarm_trial_count",
    "correct_reject_trial_count",
    "hit_rate",
    "false_alarm_rate",
    "d_prime",
    "criterion",
    "engaged_fraction",
    "minority_count",
    "descriptive_candidate"
]

display(
    candidate_sessions[
        candidate_columns
    ].head(20)
)

In [ ]:
if "CACHE_DIR" in globals():
    output_directory = (
        Path(CACHE_DIR)
        / "general_descriptive_outputs"
    )
else:
    output_directory = (
        Path.cwd()
        / "general_descriptive_outputs"
    )

output_directory.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Saving results to:",
    output_directory
)

In [ ]:
overall_summary.to_csv(
    output_directory
    / "overall_dataset_summary.csv"
)

cohort_summary.to_csv(
    output_directory
    / "vip_visp_familiar_active_summary.csv"
)

experiment_enriched.to_csv(
    output_directory
    / "experiment_metadata_with_cell_counts.csv"
)

cohort_behavior.to_csv(
    output_directory
    / "vip_visp_behavior_session_summary.csv",
    index=False
)

cohort_behavior_describe.to_csv(
    output_directory
    / "behavior_descriptive_statistics.csv"
)

candidate_sessions[
    candidate_columns
].to_csv(
    output_directory
    / "representative_session_candidates.csv",
    index=False
)

print("All tables saved.")